In [2]:
import numpy as np                                     # linear algebra
import pandas as pd                                    # data processing, CSV file I/O (e.g. pd.read_csv)
import copy                                            #to copy list
from sklearn.model_selection import train_test_split   #to split dataset into train and test set
from sklearn.svm import SVC                            #to create svc instance
from sklearn.metrics import classification_report      #to create report for precision,recall,f1-score,accuracy
from sklearn import metrics                            #to get accuracy
from sklearn.model_selection import GridSearchCV
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv #to optimise the hyper-parameter

In [3]:
df = pd.read_csv(r"H:\\THESIS\\DATASET\\2022-08-03-ss.cleaned.csv")
df.head()

,pdb_id,chain_code,seq,sst8,sst3,len,has_nonstd_aa
0,1A30,C,EDL,CBC,CEC,3,False
1,1B05,B,KCK,CBC,CEC,3,False
2,1B0H,B,KAK,CBC,CEC,3,False
3,1B1H,B,KFK,CBC,CEC,3,False
4,1B2H,B,KAK,CBC,CEC,3,False


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 477153 entries, 0 to 477152
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   pdb_id         477153 non-null  object
 1   chain_code     477153 non-null  object
 2   seq            477153 non-null  object
 3   sst8           477153 non-null  object
 4   sst3           477153 non-null  object
 5   len            477153 non-null  int64 
 6   has_nonstd_aa  477153 non-null  bool  
dtypes: bool(1), int64(1), object(5)
memory usage: 22.3+ MB


In [5]:
maxlen_seq = 128
input_seqs, target_seqs = df[['seq', 'sst8']][(df.len <= maxlen_seq) & (~df.has_nonstd_aa)].values.T
#input_grams = seq2ngrams(input_seqs)
print(input_seqs[0:5])
print(input_seqs.size)

['EDL' 'KCK' 'KAK' 'KFK' 'KAK']
103801


In [6]:
print(target_seqs[0:5])
print(target_seqs.size)

['CBC' 'CBC' 'CBC' 'CBC' 'CBC']
103801


In [7]:
for row in range(len(target_seqs)):
    secondary_lenth = len(target_seqs[row])
    primary_lenth = len(input_seqs[row])
    
    if(secondary_lenth != primary_lenth):
        print("(",row,") Secondary_Structure ->", target_seqs[row]," Primary_Structure -> ",input_seqs[row])

print("OKAY")

OKAY


In [8]:
secondary_count = 0
primary_count = 0
for row in range(len(target_seqs)):
    secondary_lenth = len(target_seqs[row])
    primary_lenth = len(input_seqs[row])
    secondary_count = secondary_count + secondary_lenth
    primary_count = primary_count + primary_lenth
    if(secondary_lenth != primary_lenth):
        print("(",row,") Secondary_Structure ->", target_seqs[row]," Primary_Structure -> ",input_seqs[row])

print("count of secondary structure : ",secondary_count)
print("count of primary structure : ",primary_count)

count of secondary structure :  8386644
count of primary structure :  8386644


In [9]:
def split(sequence):
    return [char for char in sequence]

In [10]:
primary_split = []
secondary_split = []
for row in range(int(len(target_seqs)/40)):
    primary_split.append(split(input_seqs[row]))
    secondary_split.append(split(target_seqs[row]))


In [11]:
primary_split

[['E', 'D', 'L'],
 ['K', 'C', 'K'],
 ['K', 'A', 'K'],
 ['K', 'F', 'K'],
 ['K', 'A', 'K'],
 ['K', 'M', 'K'],
 ['K', 'H', 'K'],
 ['K', 'I', 'K'],
 ['K', 'A', 'K'],
 ['K', 'G', 'K'],
 ['K', 'G', 'K'],
 ['K', 'F', 'K'],
 ['K', 'P', 'K'],
 ['K', 'A', 'K'],
 ['K', 'D', 'K'],
 ['K', 'S', 'K'],
 ['K', 'T', 'K'],
 ['K', 'Y', 'K'],
 ['K', 'A', 'K'],
 ['K', 'N', 'K'],
 ['K', 'Q', 'K'],
 ['K', 'V', 'K'],
 ['K', 'L', 'K'],
 ['F', 'P', 'R'],
 ['K', 'L', 'K'],
 ['M', 'A', 'S'],
 ['M', 'A', 'S'],
 ['M', 'A', 'S'],
 ['M', 'A', 'S'],
 ['M', 'A', 'S'],
 ['M', 'A', 'S'],
 ['A', 'P', 'R'],
 ['G', 'A', 'R'],
 ['K', 'A', 'A'],
 ['K', 'A', 'A'],
 ['K', 'A', 'A'],
 ['K', 'A', 'A'],
 ['K', 'A', 'A'],
 ['K', 'A', 'A'],
 ['G', 'A', 'R'],
 ['G', 'A', 'K'],
 ['G', 'A', 'K'],
 ['G', 'A', 'R'],
 ['K', 'A', 'A'],
 ['K', 'A', 'A'],
 ['K', 'A', 'A'],
 ['K', 'A', 'A'],
 ['K', 'A', 'K'],
 ['K', 'E', 'K'],
 ['K', 'W', 'K'],
 ['Q', 'N', 'W'],
 ['Q', 'Q', 'W'],
 ['Q', 'K', 'W'],
 ['M', 'L', 'F'],
 ['F', 'P', 'R'],
 ['K', 'R'

In [12]:
secondary_split

[['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B', 'C'],
 ['C', 'B'

In [13]:
input_seqs[1]

'KCK'

In [14]:
# Split into training and test sets (80% train, 20% test)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Further split the training set into training and validation sets (80% train, 20% validation)
train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42)

In [15]:
train_df

,pdb_id,chain_code,seq,sst8,sst3,len,has_nonstd_aa
386615,4XYE,A,MAKVLCVLYDDPTSGYPPLYARNAIPKIERYPDGQTVPNPKHIDFV...,CCEEEEECBCCCTTCSSCCCSCSCCCCCCBCTTSCBCCCCSCCSSC...,CCEEEEECECCCCCCCCCCCCCCCCCCCCECCCCCECCCCCCCCCC...,391,False
366645,5VJD,A,SKIFDFVKPGVITGDDVQKVFQVAKENNFALPAVNCVGTDSINAVL...,CCGGGTCCSEECCTHHHHHHHHHHHHTTCCEEEEECCSHHHHHHHH...,CCHHHCCCCEECCCHHHHHHHHHHHHCCCCEEEEECCCHHHHHHHH...,358,False
181969,3TY6,C,SNAMGNFHATTIFAVHHNGECAMAGDGQVTMGNAVVMKHTARKVRK...,CCCCCCCCCCCEEEEEETTEEEEEEECCEEETTTEEEESCCCCEEE...,CCCCCCCCCCCEEEEEECCEEEEEEECCEEECCCEEEECCCCCEEE...,183,False
472288,6X2W,C,GGSMEGILDFSNDLDIALLDQVVSTFYQGSGVQQKQAQEILTKFQD...,CCCCTTTTCTTSCCCHHHHHHHHHHHHTCCHHHHHHHHHHHHHHHT...,CCCCCCCCCCCCCCCHHHHHHHHHHHHCCCHHHHHHHHHHHHHHHC...,1024,False
34383,4RTW,C,GSHMTFVALYDYVSRTETDLSFKKGERLQIVNNTEGDWWLAHSLTT...,CCCCCEEESSCBCCCSSSBCCBCTTCEEEEEECCSSSEEEEEETTT...,CCCCCEEECCCECCCCCCECCECCCCEEEEEECCCCCEEEEEECCC...,61,False
...,...,...,...,...,...,...,...
372667,4R51,B,MGYTVAVVGATGAVGAQMIKMLEESTLPIDKIRYLASARSAGKSLK...,CCEEEEEETTTSHHHHHHHHHHHTCSSCEEEEEEEECTTTTTCEEE...,CCEEEEEECCCCHHHHHHHHHHHCCCCCEEEEEEEECCCCCCCEEE...,366,False
466927,6NYM,A,AFFTTVIIPAIVGGIATGTAVGTVSGLLGWGLKQAEEANKTPDKPD...,CCCCCCCCCCCCCCCCCCCCCCCCCCCCCHHHHHHHSSSCCCCCCB...,CCCCCCCCCCCCCCCCCCCCCCCCCCCCCHHHHHHHCCCCCCCCCE...,821,False
300752,4DJ5,X,AAQTNAPWGLARISSTSPGTSTYYYDESAGQGSCVYVIDTGIEASH...,CEETTCCHHHHHHTCSSSSCCCEECCTTTTTTEEEEEEESCCCTTC...,CEECCCCHHHHHHCCCCCCCCCEECCCCCCCCEEEEEEECCCCCCC...,279,False
356367,4XDZ,A,MAKIYKDEDISLEPIKNKTIAILGYGSQGRAWALNLRDSGLNVVVG...,CCCEECGGGCCSGGGTTCEEEEECCSHHHHHHHHHHHHTTCEEEEE...,CCCEECHHHCCCHHHCCCEEEEECCCHHHHHHHHHHHHCCCEEEEE...,343,False


In [16]:
#pip install datasets

In [17]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch
from datasets import Dataset

# Load the pre-trained ProtBERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('Rostlab/prot_bert')
model = BertForSequenceClassification.from_pretrained('Rostlab/prot_bert')  # Adjust num_labels as per your task

# Define a function to tokenize the protein sequences
def tokenize_function(examples):
    return tokenizer(examples['seq'], padding="max_length", truncation=True)


# Convert to Hugging Face dataset
dataset = Dataset.from_pandas(df)

# Apply tokenization
tokenized_dataset = dataset.map(tokenize_function, batched=True)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at Rostlab/prot_bert and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/477153 [00:00<?, ? examples/s]

Asking to pad to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no padding.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [18]:
tokenized_dataset

Dataset({
    features: ['pdb_id', 'chain_code', 'seq', 'sst8', 'sst3', 'len', 'has_nonstd_aa', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 477153
})

In [19]:
tokenized_dataset_q3 = dataset.map(lambda examples: tokenizer(examples['sst3'], padding="max_length", truncation=True), batched=True)
tokenized_dataset_q8 = dataset.map(lambda examples: tokenizer(examples['sst8'], padding="max_length", truncation=True), batched=True)

Map:   0%|          | 0/477153 [00:00<?, ? examples/s]

Map:   0%|          | 0/477153 [00:00<?, ? examples/s]

In [20]:
tokenized_dataset

Dataset({
    features: ['pdb_id', 'chain_code', 'seq', 'sst8', 'sst3', 'len', 'has_nonstd_aa', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 477153
})

In [21]:
#%pip install tf-keras

In [22]:
from transformers import RobertaForSequenceClassification
from torch.utils.data import DataLoader
import torch
from transformers import Trainer, TrainingArguments

# Define a custom model that can handle both tasks
class MultiTaskModel(torch.nn.Module):
    def __init__(self, base_model, num_labels_1, num_labels_2):
        super(MultiTaskModel, self).__init__()
        self.base_model = base_model
        # Define two classification heads
        self.classifier_q3 = torch.nn.Linear(self.base_model.config.hidden_size, num_labels_1)
        self.classifier_q8 = torch.nn.Linear(self.base_model.config.hidden_size, num_labels_2)

    def forward(self, input_ids, attention_mask=None):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state
        # Use the hidden states to predict both tasks
        logits_q3 = self.classifier_q3(hidden_states[:, 0, :])  # Use [CLS] token for classification
        logits_q8 = self.classifier_q8(hidden_states[:, 0, :])
        return logits_q3, logits_q8


In [23]:
# Load ProtBERT or any base model
base_model = RobertaForSequenceClassification.from_pretrained('Rostlab/prot_bert')  # Adjust num_labels based on your tasks

# Initialize MultiTask Model
multi_task_model = MultiTaskModel(base_model, num_labels_1=3, num_labels_2=8)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at Rostlab/prot_bert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight', 'embeddings.LayerNorm.bias', 'embeddings.LayerNorm.weight', 'embeddings.position_embeddings.weight', 'embeddings.token_type_embeddings.weight', 'embeddings.word_embeddings.weight', 'encoder.layer.0.attention.output.LayerNorm.bias', 'encoder.layer.0.attention.output.LayerNorm.weight', 'encoder.layer.0.attention.output.dense.bias', 'encoder.layer.0.attention.output.dense.weight', 'encoder.layer.0.attention.self.key.bias', 'encoder.layer.0.attention.self.key.weight', 'encoder.layer.0.attention.self.query.bias', 'encoder.layer.0.attention.self.query.weight', 'encoder.layer.0.attention.self.value.bias', 'encoder.layer.0.attention.self.value.weight', 'encoder.layer.0.intermediate.dense.bias', 'encoder.layer.0.intermediate.dense.weight', 'enc

In [24]:
#%pip install transformers[torch]

In [25]:
%pip install 'accelerate>=0.26.0

Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: "'accelerate"


In [27]:
# Trainer setup
training_args = TrainingArguments(
    output_dir='./results',          # output directory
    num_train_epochs=3,              # number of training epochs
    per_device_train_batch_size=8,   # batch size for training
    per_device_eval_batch_size=8,    # batch size for evaluation
    warmup_steps=500,                # number of warmup steps for learning rate scheduler
    weight_decay=0.01,               # strength of weight decay
    logging_dir='./logs',            # directory for storing logs
)

trainer = Trainer(
    model=multi_task_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,  # You can use a separate validation set
)

class MultiTaskModel(torch.nn.Module):
    def __init__(self, base_model, num_labels_1, num_labels_2):
        super(MultiTaskModel, self).__init__()
        self.base_model = base_model
        # Define two classification heads
        self.classifier_q3 = torch.nn.Linear(self.base_model.config.hidden_size, num_labels_1)
        self.classifier_q8 = torch.nn.Linear(self.base_model.config.hidden_size, num_labels_2)

    def forward(self, input_ids, attention_mask=None):
        # Pass `output_hidden_states=True` to get hidden states
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
        hidden_states = outputs.hidden_states[-1]  # Get the last hidden state from the model
        # Use the hidden states to predict both tasks
        logits_q3 = self.classifier_q3(hidden_states[:, 0, :])  # Use [CLS] token for classification
        logits_q8 = self.classifier_q8(hidden_states[:, 0, :])
        return logits_q3, logits_q8


# Train the model
trainer.train()

AttributeError: 'SequenceClassifierOutput' object has no attribute 'last_hidden_state'